# Cybersecurity ML Project: Step-by-Step Walkthrough

This notebook is organized as a complete machine learning workflow.

1. Import required libraries
2. Load and inspect the dataset
3. Clean and preprocess the data
4. Explore data patterns with visualizations
5. Build baseline and advanced ML models
6. Tune models with cross-validation
7. Evaluate using accuracy, F1-score, confusion matrix, and classification report
8. Save the best model and metadata

In [ ]:
# ===============================
# 📦 Importing Libraries
# ===============================

# 🚀 Data Manipulation and Analysis
import pandas as pd  # DataFrame operations
import numpy as np   # Numerical computations

# 📊 Data Visualization
import matplotlib.pyplot as plt  # Plotting static graphs
import seaborn as sns            # Advanced data visualization
import plotly.express as px      # Interactive visualizations
import plotly.graph_objects as go  # Custom interactive plots

In [ ]:
# 🛠 Utility Tools
import warnings
warnings.filterwarnings('ignore')  # Suppress warnings for cleaner outputs

In [ ]:
from pathlib import Path

csv_candidates = [
    Path("cyberfeddefender_dataset 2.csv"),
    Path("Capstone_Project") / "cyberfeddefender_dataset 2.csv",
]

csv_path = next((p for p in csv_candidates if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError("Could not find 'cyberfeddefender_dataset 2.csv'.")

data = pd.read_csv(csv_path)
data.columns = data.columns.str.strip()
print(f"Loaded dataset from: {csv_path.resolve()}")
print(f"Dataset shape: {data.shape}")
print(data.head())

In [ ]:
# ===============================
# 🧐 Exploring the Dataset
# ===============================

# Display basic information about the dataset
print("\n🔍 Dataset Info:")
print(data.info())

In [ ]:
# Display the first few rows for a quick preview
print("\n📋 First 5 Rows:")
print(data.head())

In [ ]:
# Display basic statistics for numerical columns
print("\n📊 Dataset Statistics:")
print(data.describe(include='all'))

In [ ]:
# ===============================
# 🔍 Checking for Missing Data
# ===============================

# Display the count of missing values for each column
missing_data = data.isnull().sum()
print("\n🔍 Missing Data Summary:")
print(missing_data[missing_data > 0])  # Display only columns with missing values

In [ ]:
# Visualizing missing data (optional for better insight)
plt.figure(figsize=(10, 6))
sns.heatmap(data.isnull(), cbar=False, cmap='viridis')
plt.title('Missing Data Heatmap')
plt.show()

In [ ]:
# ===============================
# ⚖️ Homogeneity Testing
# ===============================

# Importing necessary libraries for statistical tests
from scipy.stats import levene

In [ ]:
# Selecting numerical columns for homogeneity testing
numerical_columns = data.select_dtypes(include=['float64', 'int64']).columns

In [ ]:
# Performing Levene's test for homogeneity
print("\n⚖️ Homogeneity Test Results:")
for column in numerical_columns:
    stat, p_value = levene(data[column].dropna(), data[numerical_columns[0]].dropna())
    print(f"Column: {column} | Statistic: {stat:.3f} | P-value: {p_value:.3f}")

    if p_value < 0.05:
        print(f"❌ {column} does not have homogeneous variance.")
    else:
        print(f"✅ {column} passes the homogeneity test.")

In [ ]:
# ===============================
# 📌 Next Step
# ===============================
# Proceed with data normalization or feature engineering based on test results.
print("\n✅ Missing data handling and homogeneity testing completed!")

In [ ]:
# ===============================
# ⚖️ Normalizing Features
# ===============================

# Columns with non-homogeneous variance
columns_to_normalize = [
'Duration','Source_Port','Destination_Port',
'Flow_Packets/s','Avg_Packet_Size',
'Total_Fwd_Packets','Total_Bwd_Packets',
'Fwd_Header_Length','Inbound'
]

In [ ]:

# Applying log transformation to stabilize variance
for col in columns_to_normalize:
    data[col] = np.log1p(data[col])  # log1p handles log(0) safely

In [ ]:

# ===============================
# 🔧 Scaling Features
# ===============================

# Importing MinMaxScaler for scaling
from sklearn.preprocessing import MinMaxScaler

In [ ]:
# Initialize scaler
scaler = MinMaxScaler()

In [ ]:
# Selecting numerical columns for scaling
numerical_columns = data.select_dtypes(include=['float64', 'int64']).columns


In [ ]:
# Scaling the numerical columns
data[numerical_columns] = scaler.fit_transform(data[numerical_columns])

In [ ]:
# ===============================
# ✅ Verification
# ===============================

# Display the first few rows of the normalized and scaled dataset
print("\n📋 Transformed Dataset (First 5 Rows):")
print(data.head())

In [ ]:
# Check if the scaling was successful
print("\n📊 Summary Statistics After Scaling:")
print(data[numerical_columns].describe())

In [ ]:
# ===============================
# 📌 Next Step
# ===============================
# Proceed to feature engineering or model training.
print("\n✅ Features normalized and scaled successfully!")

In [ ]:
# ===============================
# 🔍 Dataset Overview
# ===============================

# Summary statistics for all columns
print("\n📊 Complete Summary Statistics:")
print(data.describe(include='all'))

In [ ]:
# Visualizing distributions for numerical features
numerical_columns = data.select_dtypes(include=['float64', 'int64']).columns

for col in numerical_columns:
    plt.figure(figsize=(10, 6))
    sns.histplot(data[col], kde=True, bins=30, color='blue')
    plt.title(f'Distribution of {col}')
    plt.xlabel(col)
    plt.ylabel('Frequency')
    plt.show()

In [ ]:
# ===============================
# 📈 Feature Correlation Analysis (Fixed)
# ===============================

# Selecting only numerical columns
numerical_columns = data.select_dtypes(include=['float64', 'int64']).columns
correlation_matrix = data[numerical_columns].corr()

# Visualizing the correlation matrix
plt.figure(figsize=(12, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Feature Correlation Heatmap (Numerical Columns Only)')
plt.show()

In [ ]:
# ===============================
# 🔗 Advanced Categorical Insights
# ===============================

categorical_columns = data.select_dtypes(include=['object']).columns

for col in categorical_columns:
    print(f"\n🗂 Unique Values in {col}:")
    print(data[col].value_counts())

    plt.figure(figsize=(12, 6))
    sns.countplot(data=data, y=col, order=data[col].value_counts().index, palette="viridis")
    plt.title(f'Distribution of {col}')
    plt.xlabel('Count')
    plt.ylabel(col)
    plt.show()

In [ ]:
# ===============================
# ⚠️ Advanced Anomaly Detection
# ===============================

# Using Z-score for outlier detection
from scipy.stats import zscore

for col in numerical_columns:
    z_scores = zscore(data[col])
    outliers = data[np.abs(z_scores) > 3]
    print(f"\n⚠️ Outliers detected in {col}: {len(outliers)}")

    # Visualizing outliers
    plt.figure(figsize=(10, 6))
    sns.boxplot(data=data, x=col, color="red")
    plt.title(f'Outliers in {col}')
    plt.show()

In [ ]:
# ===============================
# 📌 Insights and Next Steps
# ===============================
print("\n✅ Advanced data exploration completed! Use these insights to guide feature engineering and model selection.")

In [ ]:

# ===============================
# 📊 Descriptive Statistics
# ===============================

print("\n📈 Descriptive Statistics for Numerical Features:")
print(data.describe())

# Visualizing skewness and kurtosis for numerical features
from scipy.stats import skew, kurtosis

for col in numerical_columns:
    col_skew = skew(data[col].dropna())
    col_kurt = kurtosis(data[col].dropna())
    print(f"\n🔹 Feature: {col}")
    print(f"   Skewness: {col_skew:.3f} (Distribution asymmetry)")
    print(f"   Kurtosis: {col_kurt:.3f} (Tail heaviness)")

    # Histogram and KDE plot
    plt.figure(figsize=(10, 6))
    sns.histplot(data[col], kde=True, bins=30, color='green')
    plt.title(f'Distribution with Skewness and Kurtosis - {col}')
    plt.xlabel(col)
    plt.ylabel('Frequency')
    plt.show()

In [ ]:
# ===============================
# ⚠️ Advanced Outlier Detection
# ===============================

# Using IQR method for detailed outlier analysis
for col in numerical_columns:
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = data[(data[col] < (Q1 - 1.5 * IQR)) | (data[col] > (Q3 + 1.5 * IQR))]
    print(f"\n⚠️ Outliers in {col}: {len(outliers)}")

    # Boxplot for visualizing outliers
    plt.figure(figsize=(10, 6))
    sns.boxplot(data=data, x=col, color='red')
    plt.title(f'Outliers Analysis - {col}')
    plt.show()

In [ ]:
# ===============================
# ⭐ Feature Importance (Statistical)
# ===============================
# # Using correlation to identify top features
top_features = correlation_matrix.abs().mean().sort_values(ascending=False).index[:5]
print("\n⭐ Top Features Based on Correlation:")
print(top_features)

In [ ]:

# Visualizing the most important features
plt.figure(figsize=(12, 6))
sns.heatmap(correlation_matrix.loc[top_features, top_features], annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Top Feature Correlation Heatmap')
plt.show()

In [ ]:
# ===============================
# 📌 Next Step
# ===============================
# Use these insights to finalize feature selection and proceed to model training.
print("\n✅ Statistical analysis completed! Ready for feature engineering or model training.")

In [ ]:
# ===============================
# 🛠 Feature Creation
# ===============================

# 1️⃣ Total traffic feature
data['Total_Traffic'] = data['Bytes_Sent'] + data['Bytes_Received']
print("✅ New feature 'Total_Traffic' created.")

# 2️⃣ Packet ratio feature
data['Packet_Ratio'] = data['Total_Fwd_Packets'] / (data['Total_Bwd_Packets'] + 1)
print("✅ New feature 'Packet_Ratio' created.")

# 3️⃣ Traffic rate feature
data['Traffic_Rate'] = data['Flow_Bytes/s'] / (data['Flow_Packets/s'] + 1)
print("✅ New feature 'Traffic_Rate' created.")

# 4️⃣ Packet efficiency feature
data['Packet_Efficiency'] = data['Bytes_Sent'] / (data['Packet_Length'] + 1)
print("✅ New feature 'Packet_Efficiency' created.")

In [ ]:
# ===============================
# 🔧 Feature Transformation
# ===============================

import numpy as np

# Features that usually have skewed distribution
skewed_features = ['Bytes_Sent', 'Bytes_Received', 'Total_Traffic', 'Flow_Bytes/s']

for col in skewed_features:
    data[f'{col}_log'] = np.log1p(data[col])   # log1p handles zero values
    print(f"✅ Log-transformed feature '{col}_log' created.")

In [ ]:
# ===============================
# 🔗 Feature Encoding
# ===============================

from sklearn.preprocessing import LabelEncoder

# Find categorical columns
categorical_columns = data.select_dtypes(include=['object']).columns

# Encode them
for col in categorical_columns:
    encoder = LabelEncoder()
    data[col + '_encoded'] = encoder.fit_transform(data[col].astype(str))
    print(f"✅ Categorical feature '{col}' encoded.")

In [ ]:
# ===============================
# ⭐ Feature Selection (Fixed)
# ===============================

# Identifying features with low variance
from sklearn.feature_selection import VarianceThreshold

# Selecting numerical columns for variance thresholding
numerical_data = data.select_dtypes(include=['float64', 'int64'])

# Identifying features with low variance
low_variance_filter = VarianceThreshold(threshold=0.01)
low_variance_filter.fit(numerical_data)

# Mapping low variance features to their original dataset columns
low_variance_features = numerical_data.columns[~low_variance_filter.get_support()]
print(f"\n⚠️ Low variance features detected and removed: {list(low_variance_features)}")

# Dropping low variance features from the original dataset
data.drop(columns=low_variance_features, inplace=True, errors='ignore')

In [ ]:

# Removing redundant features (e.g., original features replaced by transformed ones)
redundant_features = ['bytes_in', 'bytes_out', 'traffic_intensity']
data.drop(columns=redundant_features, inplace=True, errors='ignore')
print("✅ Redundant features removed.")

In [ ]:
# ===============================
# 📋 Final Dataset Summary
# ===============================

print("\n📋 Final Dataset Features:")
print(data.columns)

print("\n🔢 Final Dataset Shape:", data.shape)

In [ ]:

# ===============================
# 🔧 Verification
# ===============================

print("\n📋 Final Dataset After Removing Low Variance Features:")
print(data.info())

In [ ]:
# ===============================
# 📌 Next Step
# ===============================
# Proceed to splitting the dataset into training and testing sets for model training.
print("\n✅ Feature engineering completed! Ready for model training.")

In [ ]:
# ===============================
# 📋 Step 1: Column Validation and Dataset Overview
# ===============================
print("\n📋 Available Columns in Dataset:")
print(data.columns)

In [ ]:
# Identify numerical and categorical columns
numerical_features = data.select_dtypes(include=['float64', 'int64']).columns.tolist()
categorical_features = data.select_dtypes(include=['object']).columns.tolist()

print(f"\n🔢 Numerical Features: {numerical_features}")
print(f"🗂️ Categorical Features: {categorical_features}")

## Machine Learning 



In [ ]:
# Step 1: Import ML libraries
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix
import pandas as pd
import numpy as np
import joblib

RANDOM_STATE = 42
print("ML libraries loaded (non-boosting setup).")


In [ ]:
# Step 2: Build clean binary target from Attack_Type
if "data" not in globals():
    raise NameError("Run data loading cells first.")

df_ml = data.copy()
df_ml.columns = df_ml.columns.str.strip()

if "Attack_Type" not in df_ml.columns:
    raise ValueError("Expected Attack_Type column in dataset.")

y_ml = (df_ml["Attack_Type"].astype(str).str.strip().str.lower() != "normal").astype(int)
print("Target created: 0 = Normal, 1 = Attack")
print(y_ml.value_counts(normalize=True).sort_index().round(4))


In [ ]:
# Step 3: Remove leakage columns from features
forbidden_tokens = ("attack_type", "label")
leakage_cols = [
    c for c in df_ml.columns
    if c in {"Attack_Type", "Label"} or any(tok in c.lower() for tok in forbidden_tokens)
]
X_ml = df_ml.drop(columns=leakage_cols, errors="ignore")

remaining = [c for c in X_ml.columns if any(tok in c.lower() for tok in forbidden_tokens)]
if remaining:
    raise ValueError(f"Leakage-like columns still present: {remaining}")

tmp = X_ml.copy()
tmp["__target__"] = y_ml.values
rows_before = len(tmp)
tmp = tmp.drop_duplicates().reset_index(drop=True)
rows_after = len(tmp)
y_ml = tmp.pop("__target__").astype(int)
X_ml = tmp

print(f"Leakage columns removed: {sorted(leakage_cols)}")
print(f"Rows before/after dedup: {rows_before} -> {rows_after}")
print(f"Feature shape: {X_ml.shape}")


In [ ]:
# Step 4: Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_ml, y_ml, test_size=0.2, random_state=RANDOM_STATE, stratify=y_ml
)
print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")
print(f"Train attack ratio: {y_train.mean():.4f}")
print(f"Test attack ratio: {y_test.mean():.4f}")


In [ ]:
# Step 5: Build preprocessing pipeline
numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])
print(f"Numeric features: {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")


In [ ]:
# Step 6: Compare multiple non-boosting baseline models with cross-validation
model_candidates = {
    "LogisticRegression": LogisticRegression(
        max_iter=3000, class_weight="balanced", solver="saga", random_state=RANDOM_STATE
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=500, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
    ),
    "ExtraTrees": ExtraTreesClassifier(
        n_estimators=500, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
    ),
    "DecisionTree": DecisionTreeClassifier(
        class_weight="balanced", random_state=RANDOM_STATE
    ),
    "KNN": KNeighborsClassifier(),
    "SVC_RBF": SVC(
        kernel="rbf", class_weight="balanced", random_state=RANDOM_STATE
    ),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "accuracy": "accuracy",
    "f1_weighted": "f1_weighted",
    "precision_weighted": "precision_weighted",
    "recall_weighted": "recall_weighted",
}

cv_rows = []
for model_name, model_obj in model_candidates.items():
    pipe = Pipeline([("preprocess", preprocessor), ("model", model_obj)])
    scores = cross_validate(
        pipe,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        error_score="raise",
    )
    cv_rows.append({
        "Model": model_name,
        "CV_Accuracy": scores["test_accuracy"].mean(),
        "CV_F1_weighted": scores["test_f1_weighted"].mean(),
        "CV_Precision_weighted": scores["test_precision_weighted"].mean(),
        "CV_Recall_weighted": scores["test_recall_weighted"].mean(),
    })

cv_results = pd.DataFrame(cv_rows).sort_values("CV_Accuracy", ascending=False).reset_index(drop=True)
best_model_name = cv_results.loc[0, "Model"]

print("Cross-validation results (sorted by accuracy):")
print(cv_results.round(4).to_string(index=False))
print(f"Best baseline model by CV accuracy: {best_model_name}")


In [ ]:
# Step 6A: Hyperparameter tuning (non-boosting only)
from scipy.stats import randint, loguniform
from sklearn.model_selection import RandomizedSearchCV

tuning_spaces = {
    "LogisticRegression": {
        "model__C": loguniform(1e-3, 1e2),
        "model__penalty": ["l1", "l2"],
        "model__solver": ["saga"],
        "model__max_iter": [3000],
    },
    "RandomForest": {
        "model__n_estimators": randint(300, 1400),
        "model__max_depth": [None, 10, 20, 30, 40],
        "model__min_samples_split": randint(2, 20),
        "model__min_samples_leaf": randint(1, 10),
        "model__max_features": ["sqrt", "log2", None],
        "model__class_weight": [None, "balanced", "balanced_subsample"],
    },
    "ExtraTrees": {
        "model__n_estimators": randint(300, 1400),
        "model__max_depth": [None, 10, 20, 30, 40],
        "model__min_samples_split": randint(2, 20),
        "model__min_samples_leaf": randint(1, 10),
        "model__max_features": ["sqrt", "log2", None],
        "model__class_weight": [None, "balanced"],
    },
    "DecisionTree": {
        "model__max_depth": [None, 5, 10, 20, 30, 40],
        "model__min_samples_split": randint(2, 30),
        "model__min_samples_leaf": randint(1, 15),
        "model__max_features": [None, "sqrt", "log2"],
        "model__class_weight": [None, "balanced"],
    },
    "KNN": {
        "model__n_neighbors": randint(3, 41),
        "model__weights": ["uniform", "distance"],
        "model__metric": ["minkowski", "manhattan", "euclidean"],
        "model__p": [1, 2],
    },
    "SVC_RBF": {
        "model__C": loguniform(1e-2, 1e2),
        "model__gamma": ["scale", "auto"],
        "model__kernel": ["rbf"],
    },
}

# Tune top 4 baseline models to improve runtime and still maximize final accuracy
top_models_to_tune = cv_results["Model"].head(4).tolist()
tuned_rows = []
tuned_models = {}

for model_name in top_models_to_tune:
    model_obj = model_candidates[model_name]
    search_pipe = Pipeline([("preprocess", preprocessor), ("model", model_obj)])
    search = RandomizedSearchCV(
        estimator=search_pipe,
        param_distributions=tuning_spaces[model_name],
        n_iter=25,
        scoring="accuracy",
        cv=cv,
        n_jobs=-1,
        random_state=RANDOM_STATE,
        refit=True,
        error_score="raise",
        verbose=0,
    )
    search.fit(X_train, y_train)

    tuned_models[model_name] = {
        "estimator": search.best_estimator_,
        "best_cv_accuracy": float(search.best_score_),
        "best_params": search.best_params_,
    }
    tuned_rows.append({
        "Model": model_name,
        "Tuned_CV_Accuracy": search.best_score_,
    })

tuned_results = pd.DataFrame(tuned_rows).sort_values("Tuned_CV_Accuracy", ascending=False).reset_index(drop=True)
tuned_best_name = tuned_results.loc[0, "Model"]
tuned_best_estimator = tuned_models[tuned_best_name]["estimator"]

print(f"Models tuned: {top_models_to_tune}")
print("Tuned CV accuracy results:")
print(tuned_results.round(4).to_string(index=False))
print(f"\nBest tuned model: {tuned_best_name}")
print("Best tuned params:")
print(tuned_models[tuned_best_name]["best_params"])

In [ ]:
# Step 6B: Compare baseline vs tuned CV and choose final training path
baseline_best_cv_acc = float(cv_results.loc[cv_results["Model"] == best_model_name, "CV_Accuracy"].iloc[0])
tuned_best_cv_acc = float(tuned_results.loc[0, "Tuned_CV_Accuracy"])

print(f"Baseline best CV Accuracy: {baseline_best_cv_acc:.4f} ({best_model_name})")
print(f"Tuned best CV Accuracy: {tuned_best_cv_acc:.4f} ({tuned_best_name})")

use_tuned_model = tuned_best_cv_acc >= baseline_best_cv_acc
if use_tuned_model:
    print("Using tuned model for final test evaluation.")
else:
    print("Using baseline model for final test evaluation.")

In [ ]:
# Step 7: Train final model (tuned if better, else baseline)
if "use_tuned_model" in globals() and use_tuned_model:
    best_model = tuned_best_estimator
    best_model_name = tuned_best_name
    print(f"Using tuned model: {best_model_name}")
else:
    best_model = Pipeline([
        ("preprocess", preprocessor),
        ("model", model_candidates[best_model_name]),
    ])
    best_model.fit(X_train, y_train)
    print(f"Using baseline model: {best_model_name}")

y_pred = best_model.predict(X_test)

In [ ]:
# Step 8: Evaluate on test set
test_accuracy = accuracy_score(y_test, y_pred)
test_bal_acc = balanced_accuracy_score(y_test, y_pred)
test_f1 = f1_score(y_test, y_pred, average="weighted")
test_precision = precision_score(y_test, y_pred, average="weighted", zero_division=0)
test_recall = recall_score(y_test, y_pred, average="weighted", zero_division=0)

print(f"Accuracy: {test_accuracy:.4f}")
print(f"Balanced Accuracy: {test_bal_acc:.4f}")
print(f"F1 weighted: {test_f1:.4f}")
print(f"Precision weighted: {test_precision:.4f}")
print(f"Recall weighted: {test_recall:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))


In [ ]:
# Step 9: Confusion matrix
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


In [ ]:
# Step 10: Save model and metadata
metadata = {
    "target_definition": "0=Normal, 1=Attack (from Attack_Type)",
    "rows_used": int(len(X_ml)),
    "feature_count": int(X_ml.shape[1]),
    "removed_leakage_columns": sorted(leakage_cols),
    "models_compared": list(model_candidates.keys()),
    "models_tuned": top_models_to_tune if "top_models_to_tune" in globals() else [],
    "best_model_name": best_model_name,
    "test_accuracy": float(test_accuracy),
    "test_balanced_accuracy": float(test_bal_acc),
    "test_f1_weighted": float(test_f1),
}

if "baseline_best_cv_acc" in globals():
    metadata["baseline_best_cv_accuracy"] = float(baseline_best_cv_acc)
if "tuned_best_cv_acc" in globals():
    metadata["tuned_best_cv_accuracy"] = float(tuned_best_cv_acc)
if "use_tuned_model" in globals():
    metadata["used_tuned_model"] = bool(use_tuned_model)
if "tuned_models" in globals() and "tuned_best_name" in globals():
    metadata["best_tuned_params"] = tuned_models[tuned_best_name]["best_params"]

joblib.dump(best_model, "best_ml_pipeline.pkl")
joblib.dump(metadata, "ml_metadata.pkl")
print("Saved: best_ml_pipeline.pkl, ml_metadata.pkl")
print("Metadata includes compared/tuned model details.")